In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import numpy as np
from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)

# Učitavanje
df = pd.read_csv("member_stats_training.csv")
df_matches = pd.read_csv("clan_matches_training.csv")

print(f"Broj igrača: {len(df)}")
print(f"Broj mečeva: {len(df_matches)}")

1. Vizualizacija podataka. Primecujem peak i bimodalnu raspodelu:
avg_stars_11 -> [3,3.5]
avg_stars_3 -> [3.5,4.2]

Verovatno default jacina igraca na pocetku.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Pripremamo tri grupe podataka
df_all = df
df_inside = df[df['avg_stars_top_3_players'].between(3.5, 4.5)]
df_outside = df[~df['avg_stars_top_3_players'].between(3.5, 4.5)]

groups = [
    (df_all, "Ceo Dataset", "skyblue"),
    (df_inside, "Inside Peak (3.5 - 4.5)", "orange"),
    (df_outside, "Outside Peak (Ostali)", "lightgreen")
]

# Uzimamo numeričke kolone
numeric_cols = df.select_dtypes(include=['number']).columns

for col in numeric_cols:
    # Kreiramo grid: 3 reda (za svaku grupu) i 2 kolone (hist i box)
    fig, axes = plt.subplots(3, 2, figsize=(15, 12))
    fig.suptitle(f'Analiza kolone: {col}', fontsize=16, fontweight='bold', y=1.02)

    for i, (data, label, color) in enumerate(groups):
        # Histogram (leva kolona)
        sns.histplot(data[col], kde=True, ax=axes[i, 0], color=color)
        axes[i, 0].set_ylabel(label, fontsize=12, fontweight='bold')
        
        # Boxplot (desna kolona)
        sns.boxplot(x=data[col], ax=axes[i, 1], color=color)
        
    plt.tight_layout()
    plt.show()

2. Inicijalni feature engineering

In [ ]:
# Mapiranje plaćanja
mapping = {
    '0) NonPayer': 0, '1) ExPayer': 1, '2) Minnow': 2,
    '3) Dolphin': 3, '4) Whale': 4, 'Negative': -1
}
df['payment_numeric'] = df['dynamic_payment_segment'].map(mapping)

# Kreiranje baznog power index-a
df['power_index'] = (df['avg_training_bonus'] / 10 + df['avg_stars_top_11_players']) 

# Grupišemo članove po klanu i sortiramo ih od 1 do 6
df_sorted = df.sort_values(['clan_id', 'power_index'], ascending=[True, False])
df_sorted['rank'] = df_sorted.groupby('clan_id').cumcount() + 1

# Zadržavamo samo top 6 igrača za svaki klan
df_members = df_sorted[df_sorted['rank'] <= 6].copy()
display(df_members.head())

3. Ideja da se simulisu sve utakmice, na osnovu skora da probamo da zakljucimo kako su bili upareni menadzeri. Ispostavilo se da je presporo i da ima previse mogucih raspodela, ujedno i premalo informacija za bilo kakvu simulaciju.

In [ ]:
def solve_global_tournament(m1, m2, total1, total2, p1, p2):
    # Mapiranje: bazni poeni -> broj remija koje taj igrač ima u 2 utakmice
    draw_map = {0: 0, 1: 1, 2: 2, 3: 0, 4: 1, 6: 0}
    possible_scores = [0, 1, 2, 3, 4, 6]

    # 1. Pronađi sve validne kombinacije za Klan 1 i Klan 2
    valid_c1 = [c for c in itertools.product(possible_scores, repeat=6) if sum(s * m for s, m in zip(c, m1)) == total1]
    valid_c2 = [c for c in itertools.product(possible_scores, repeat=6) if sum(s * m for s, m in zip(c, m2)) == total2]

    best_pair = None
    max_total_power_corr = -1e9

    # 3. Upari ih i proveri globalnu konzistentnost turnira
    for c1_scores in valid_c1:
        draws_c1 = sum(draw_map[s] for s in c1_scores)
        for c2_scores in valid_c2:
            draws_c2 = sum(draw_map[s] for s in c2_scores)
            
            # Uslov 1: Broj remija mora biti isti u oba klana
            if draws_c1 != draws_c2:
                continue
            
            # Uslov 2: Ukupna suma baznih poena mora odgovarati nuli (36 - remiji)
            if sum(c1_scores) + sum(c2_scores) == (36 - draws_c1):
                # Heuristika: biramo par sa najvećom korelacijom snage i uspeha
                corr1 = sum(p * s for p, s in zip(p1, c1_scores))
                corr2 = sum(p * s for p, s in zip(p2, c2_scores))
                
                if (corr1 + corr2) > max_total_power_corr:
                    max_total_power_corr = corr1 + corr2
                    best_pair = (c1_scores, c2_scores)
                    
    return best_pair

In [ ]:
clan_lookup = {}
for cid, group in df_members.groupby('clan_id'):
    group = group.sort_values('user_id')
    clan_lookup[cid] = {
        'user_ids': group['user_id'].values,
        'multipliers': group['clan_multiplier'].values,
        'powers': group['power_index'].values,
        'cohorts': group['cohort_day'].values,
        'segments': group['dynamic_payment_segment'].values
    }

micro_records = []
sample_matches = df_matches.head(1000) # Ubrzano za eksperiment

for idx, match in tqdm(sample_matches.iterrows(), total=len(sample_matches), desc="Dekompozicija mečeva"):
    c1_id, c2_id = match['clan_1_id'], match['clan_2_id']
    t1, t2 = match['clan_1_points'], match['clan_2_points']
    
    d1, d2 = clan_lookup.get(c1_id), clan_lookup.get(c2_id)
    
    if d1 and d2:
        solution = solve_global_tournament(
            d1['multipliers'], d2['multipliers'], t1, t2, d1['powers'], d2['powers']
        )
        
        if solution:
            c1_scores, c2_scores = solution
            for i in range(6):
                micro_records.append({
                    'match_id': idx, 'clan_id': c1_id, 'opponent_clan_id': c2_id,
                    'user_id': d1['user_ids'][i], 'power_index': d1['powers'][i],
                    'multiplier': d1['multipliers'][i], 'cohort_day': d1['cohorts'][i],
                    'payment_segment': d1['segments'][i], 'base_score': c1_scores[i]
                })
            for i in range(6):
                micro_records.append({
                    'match_id': idx, 'clan_id': c2_id, 'opponent_clan_id': c1_id,
                    'user_id': d2['user_ids'][i], 'power_index': d2['powers'][i],
                    'multiplier': d2['multipliers'][i], 'cohort_day': d2['cohorts'][i],
                    'payment_segment': d2['segments'][i], 'base_score': c2_scores[i]
                })

df_micro = pd.DataFrame(micro_records)
print(f"Uspešno popunjen df_micro sa {len(df_micro)} unosa.")
display(df_micro.head())

In [ ]:
# Priprema
le = LabelEncoder()
df_micro['target'] = le.fit_transform(df_micro['base_score'])
df_micro['segment_numeric'] = df_micro['payment_segment'].str.extract(r'(\d+)').fillna(0).astype(int)

features = ['power_index', 'multiplier', 'cohort_day', 'segment_numeric']
X = df_micro[features]
y = df_micro['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Model
model_micro = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, 
    objective='multi:softprob', random_state=42
)
model_micro.fit(X_train, y_train)
y_pred = model_micro.predict(X_val)

print(f"Accuracy na individualnim duelima: {accuracy_score(y_val, y_pred):.2f}")
print("\nDetaljan izveštaj:")
print(classification_report(y_val, y_pred, target_names=[str(c) for c in le.classes_], zero_division=0))

Feature enginner: power index(neuspelo, nije doprinelo krajnjem accuracy)

In [ ]:
# Pretraga za optimalnim a i b parametrima 
a_values = [0.1, 0.5, 1, 2, 5, 10, 20, 50]
b_values = [0.1, 0.5, 1, 2, 5, 10, 20, 50]

best_full_acc = 0
best_a = None
best_b = None

print("Započinjem super-brzu pretragu za parametre a i b... ")

y_search = df_final['clan_winner'] - 1
W_search = (df_final['clan_1_points'] - df_final['clan_2_points']).abs() + 1
train_idx, val_idx = train_test_split(df_final.index, test_size=0.25, random_state=0) 

y_tr_s, y_val_s = y_search.loc[train_idx], y_search.loc[val_idx] 
w_tr_s, w_val_s = W_search.loc[train_idx], W_search.loc[val_idx] 

base_features = ['bonus_min_ratio', 'bonus_min_diff', 'bonus_mean_ratio']

for a in a_values: 
    for b in b_values: 
        # Munjevito računanje power_index-a 
        pi_series = (member_df['avg_stars_top_11_players']/a + member_df['avg_training_bonus']/b) * member_df['clan_multiplier'] 
        pi_mean = pi_series.groupby(member_df['clan_id']).mean() 
        
        pi_c1 = df_final['clan_1_id'].map(pi_mean) 
        pi_c2 = df_final['clan_2_id'].map(pi_mean) 
        
        df_final['power_index_mean_diff'] = pi_c1 - pi_c2 
        df_final['power_index_mean_ratio'] = (pi_c1 + 0.01) / (pi_c2 + 0.01) 
        
        X_search = df_final[base_features + ['power_index_mean_diff', 'power_index_mean_ratio']] 
        X_tr_s = X_search.loc[train_idx] 
        X_val_s = X_search.loc[val_idx] 

        model_search = XGBClassifier(
            n_estimators=138, learning_rate=0.022, max_depth=4,    
            subsample=0.68, reg_alpha=0.0172, reg_lambda=0.161,
            gamma=2.182, min_child_weight=12, colsample_bytree=1,
            random_state=42, n_jobs=-1 
        ) 
        
        model_search.fit(X_tr_s, y_tr_s, sample_weight=w_tr_s) 
        val_preds_s = model_search.predict(X_val_s) 
        full_acc = accuracy_score(y_val_s, val_preds_s) 
        
        if full_acc > best_full_acc:
            best_full_acc = full_acc 
            best_a = a 
            best_b = b 
            print(f"--> Novi NAJBOLJI rezultat! a={a:5.1f}, b={b:5.1f} | Ukupni Acc: {full_acc:.4f} ")

print("=" * 50)
print(f"ZAVRŠENO! Najbolji parametri: a={best_a}, b={best_b} sa Ukupnim Accuracy-jem = {best_full_acc:.4f} ")

Mnostvo featura kreirano za koje se ispostavilo da nisu doprineli preciznosti krajnjeg modela.

In [ ]:
import pandas as pd
import numpy as np

# Pomoćna funkcija sa SVIM isprobanim metrikama i agregacijama
def aggregate_all_clan_stats(df, a=1, b=1):
    df = df.copy()
    
    # 1. Izvedene metrike za svakog igrača
    df['player_eff_power'] = df['avg_stars_top_11_players'] * (1 + df['avg_training_bonus']/10)
    df['player_exp_contrib'] = df['player_eff_power'] * df['clan_multiplier']
    df['player_bonus_active_synergy'] = df['avg_training_bonus'] * df['days_active_last_7_days']
    df['player_bonus_efficiency'] = df['avg_training_bonus'] / (df['avg_stars_top_11_players'] + 1)
    df['power_index'] = (df['avg_stars_top_11_players']/a + df['avg_training_bonus']/b)*df['clan_multiplier']

    # 2. Proširena logika grupisanja za ceo klan
    agg_logic = {
        'avg_stars_top_11_players': ['mean', 'min', 'max', 'std'],
        'avg_training_bonus': ['mean', 'min', 'max', 'std', 'median', 'skew'],
        'days_active_last_7_days': ['mean', 'min', 'max', 'std', 'skew'],
        'clan_multiplier': ['sum', 'mean', 'max', 'std'],
        'player_eff_power': ['mean', 'max'],
        'player_exp_contrib': ['mean', 'sum'],
        'player_bonus_active_synergy': ['mean'],
        'player_bonus_efficiency': ['mean'],
        'power_index': ['mean', 'sum']
    }

    clan_feats = df.groupby('clan_id').agg(agg_logic)
    
    # Sređivanje imena kolona nakon agregacije
    clan_feats.columns = [
        'stars_11_mean', 'stars_11_min', 'stars_11_max', 'stars_11_std',
        'bonus_mean', 'bonus_min', 'bonus_max', 'bonus_std', 'bonus_median', 'bonus_skew',
        'activity_7_mean', 'activity_7_min', 'activity_7_max', 'activity_7_std','activity_7_skew',
        'multiplier_sum', 'multiplier_mean', 'multiplier_max', 'multiplier_std',
        'eff_power_mean', 'eff_power_max',
        'exp_contrib_mean', 'exp_contrib_sum',
        'bonus_active_synergy', 'bonus_efficiency',
        'power_index_mean', 'power_index_sum'
    ]
    clan_feats = clan_feats.reset_index()

    # 3. Dodatne izvedene kolone disperzije kvaliteta na nivou klana
    clan_feats['bonus_mean_median_diff'] = clan_feats['bonus_mean'] - clan_feats['bonus_median']
    clan_feats['bonus_range'] = clan_feats['bonus_max'] - clan_feats['bonus_min']
    clan_feats['bonus_dropoff'] = clan_feats['bonus_mean'] - clan_feats['bonus_min']
    
    return clan_feats

# Ekstrakcija i spajanje sa mečevima
train_clan_features_all = aggregate_all_clan_stats(member_df)

df_final_all = train_matches.merge(train_clan_features_all, left_on='clan_1_id', right_on='clan_id')
df_final_all = df_final_all.merge(train_clan_features_all, left_on='clan_2_id', right_on='clan_id', suffixes=('_c1', '_c2'))

# Automatsko pravljenje DIFF i RATIO kolona za SVE generisane feature
base_stats = [c for c in train_clan_features_all.columns if c != 'clan_id']
all_feature_list = []

for col in base_stats:
    c1, c2 = f"{col}_c1", f"{col}_c2"
    df_final_all[f"{col}_diff"] = df_final_all[c1] - df_final_all[c2]
    df_final_all[f"{col}_ratio"] = (df_final_all[c1] + 0.01) / (df_final_all[c2] + 0.01)
    all_feature_list.extend([f"{col}_diff", f"{col}_ratio"])

print(f"Dataset za eksperiment je spreman sa {len(all_feature_list)} kolona.")